# Buried resistive dome: OT inversion with pure Gaussian noise (4%)

True model from `MT2DTrueModels.create_buried_dome` (parameters live in `src/mt2d_inv/models.py`):

- 100 Ω·m background half-space
- 10 Ω·m conductive overburden, about 500 m thick
- 1000 Ω·m resistive dome under the cover, top at about 300 m, 2–3 km wide
- 10 Ω·m conductive block beside it, to compare recovery of resistive and conductive anomalies
- Frequencies 1e-3–1e3 Hz (5 per decade); 41 stations at 300 m spacing

MT is sensitive to conductors and weakly sensitive to resistors, so as noise grows the resistive dome usually blurs first, which separates the behaviour of the misfits.
This notebook runs **OT** (`mode="6dot"`) at relative Gaussian noise level **0.04**.

In [ ]:
import os
import sys
import torch
from pathlib import Path
import numpy as np

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "mt2d_inv" / "__init__.py").is_file():
            return p
    raise RuntimeError("Cannot find project root (expected src/mt2d_inv/__init__.py)")

root_path = str(_find_project_root(Path.cwd()))
if root_path not in sys.path:
    sys.path.insert(0, root_path)

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

from src.mt2d_inv import MT2DInverterWeightedCost, MT2DTrueModels
from src.mt2d_inv.io import ExperimentLogger
from src.mt2d_inv.plotting import (
    plot_ot_mse_convergence,
    plot_ot_mse_convergence_from_dirs,
    plot_rho_fitting_from_npz,
    plot_ot_mse_pseudosection_from_npz,
)

logger = ExperimentLogger(
    model_tag="burieddome",
    output_root=Path("test_results"),
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on {device}")
print(f"Project root: {root_path}")

In [ ]:
# 1) Build buried-dome true model (default geometry + fine shallow grid)
yn, zn, nza, sig_true = MT2DTrueModels.create_buried_dome(
    zn=None,
    yn=None,
    nza=10,
    device=device,
)

# 2) Survey setup: 1e3 to 1e-3 Hz (5 per decade), 41 stations at 300 m spacing
freqs = torch.logspace(3, -3, 31, device=device)
stations = torch.linspace(-6000.0, 6000.0, 41, device=device)

print(f"Grid (nz, ny): {(len(zn)-1, len(yn)-1)}")
print(f"Stations: {len(stations)}, Freqs: {len(freqs)}")

In [ ]:
# 3) Inverter + synthetic data
inv = MT2DInverterWeightedCost(
    yn=torch.tensor(yn, dtype=torch.float64, device=device),
    zn=torch.tensor(zn, dtype=torch.float64, device=device),
    nza=nza,
    freqs=freqs,
    stations=stations,
    device=device,
    random_seed=123,
    te_weight=1.0,
    tm_weight=1.0,
    data_loss_scale=15000,
)
inv.set_forward_operator()
inv.sig_true = sig_true
inv.create_synthetic_data(noise_level=0.04,
                        noise_type="gaussian",
                        )

inv.initialize_model(initial_sigma=0.01)
inv.plot_initial_model(clip_to_stations=True, ylim=[4, 0], synthetic_figwidth_scale=1.0)


In [ ]:
# 4) OT inversion (6dot)
inv.initialize_model(initial_sigma=0.01)
final_sigma_ot = inv.run_inversion(
    n_epochs=300,
    mode="6dot",
    progress_interval=20,
    current_lambda=3,
    use_adaptive_lambda=True,
    lr=0.03,
    update_interval=5,
    norm_type="L2",
    alpha=0.5,
    use_depth_weights=True,
    rms_chi2_stop=1.05,
    profile_timing=False,
)
print("OT inversion done")
print(final_sigma_ot.min().item(), final_sigma_ot.max().item())

In [ ]:
# 5) Plot OT results
inv.plot_loss_history()
inv.plot_model_comparison(
    cmap="jet_r",
    clip_to_stations=True,
    ylim=[4, 0],
    vmin=1,
    vmax=3,
    synthetic_figwidth_scale=1.0,
)
inv.plot_data_fitting()
inv.plot_1d_profiles(depth_limit_km=4)
inv.plot_gradient_history()

In [ ]:
# 6) Save OT results
ot_run_dir = logger.save_from_inverter(inv, run_name="ot_0.04")
history_ot = list(inv.loss_history)
print(f"OT run saved: {ot_run_dir}")